# 3 控制流

Graph 与 Chain 最大的区别：执行路径不再固定。本章讲 LangGraph 的三种控制流手段：

1. **分支**（第 1 节）：一个节点的下游可以**并行**（1.1 静态分支）、可以**按条件选择**（1.2 条件分支）、可以在**运行时动态生成**（1.3 Send 动态分支），1.4 补充并行汇合用的 Defer 节点
2. **循环**（第 2 节）：边可以指回上游节点，反复执行直到满足退出条件——ReAct 等 agent 的核心
3. **Command**（第 3 节）：一个 Node 同时完成「更新 State」和「指定下一跳」

本章案例用到第 2 章的 `MessagesState` / `add_messages`，建议先学完第 2 章再读本章。

# 1. 分支结构

三种分支回答同一个问题：**一个节点执行完后，下游是谁、有几个？** 区别在于答案在什么时候确定：

| 类型 | 下游候选 | 决定时机 | 运行时行为 | 代码 |
|---|---|---|---|---|
| 静态分支（1.1） | 编译期画死的固定集合 | 编译期 | 所有下游**全部**并行执行（fan-out） | `add_edge` |
| 条件分支（1.2） | 编译期给定候选集合 | 编译期定候选，运行时选择 | 路由函数按 State 从候选中选 1 个或多个 | `add_conditional_edges` |
| 动态分支（1.3） | 运行时才生成 | 完全运行时 | Node 返回 Send，动态创建 N 个并行任务 | `Send` + `Command` |

一句话：**静态是「固定并行」，条件是「固定候选、运行时选择」，动态是「候选本身由运行时数据决定」**。三者互补，可以组合使用（比如 Send 发出的 worker 实例内部再走条件分支）。

## 1.1 静态分支：并行 fan-out

+ 定义：从源节点出发的**多条边在图编译阶段就全部写好**，运行时这些边**全部生效**——一个节点同时触发它的所有下游（fan-out，扇出）
+ 特点：
    - 下游节点集合在编译时确定，运行时**全部**执行，不存在「选哪条」
    - 适合并行处理多个独立任务（比如同时生成笑话和诗）
    - 想根据 State 选下游 → 1.2 的条件分支；想运行时才决定有多少个并行任务 → 1.3 的 Send
+ 核心判断：编译期画好 N 条边，运行时 N 个下游都跑 -> 静态分支

**super-step**：当多个节点都由同一个上游触发时，它们会在同一个 super-step 被激活。super-step 是 LangGraph 执行循环的基本单位 —— 「一批并行调度的节点执行完毕 + 它们的更新合并进 State」。同一 super-step 内的节点读到的 State 是本 super-step **开始前**的快照，互相看不到对方的更新（1.4 的 defer 节点就是为此设计的）。

In [ ]:
# 示例：两个 LLM 节点并行执行（标准格式）
# 标准 LLM 节点格式：读完整历史 state["messages"]，返回 {"messages": [response]}
# 两个并行节点写同一个 key（messages）→ add_messages 按 id 去重合并，两条回复都保留

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    topic: str


def joke_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」讲一个笑话")]
    )
    response.name = "joke"           # 用 name 标记角色，便于区分并行节点的输出
    return {"messages": [response]}  # 标准格式：返回消息增量，走 add_messages 合并


def poem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」写一首诗")]
    )
    response.name = "poem"
    return {"messages": [response]}


builder = StateGraph(state_schema=ChatState)
builder.add_node("joke_node", joke_node)
builder.add_node("poem_node", poem_node)
builder.add_edge(START, "joke_node")
builder.add_edge(START, "poem_node")
builder.add_edge("joke_node", END)
builder.add_edge("poem_node", END)

graph = builder.compile()
result = graph.invoke({
    "topic": "猫",
    "messages": [HumanMessage(content="请围绕这个主题展示你的才华")],
})
rprint(result)

# 显示图结构：两条静态边从 START 并行出发
display(graph)

## 1.2 条件分支

与静态分支不同，条件分支的**下游节点由运行时函数决定**：节点执行完后，LangGraph 调用一个**路由函数**读取 State，根据返回值从映射表里选出下一个节点。

+ 定义：路由函数 `(state) -> str`，返回值对应映射表里的 key，决定下一跳
+ 特点：
    - 下游候选集合在编译时通过映射表确定
    - **运行时**才决定走哪条边（路由函数读 State 判断）
    - 一个来源可以同时有多条条件边（多目标分支）
+ 与静态分支对比：
    - 静态分支（并行分支）：所有下游**同时**执行，运行前就确定
    - 条件分支：候选下游**二选一**，必须等路由函数跑完才知道去向
+ 核心判断：路由函数的返回值决定下游 -> 条件分支
+ 代码：`add_conditional_edges(源节点, 路由函数, {key: 目标节点})`

```python
# 三个参数：从哪个节点出发、路由函数、返回值 → 目标节点 的映射表
builder.add_conditional_edges("a", route, {"x": "b", "y": "c"})
```

**路由函数参数说明**：

- 签名：`def route(state) -> str`，与 Node 一样，第一个参数是**完整 State 快照**，按 key 读取任何需要的字段（全局 State、或私有 schema 中声明的同名 key），据此判断走向
- 返回值：必须是映射表的 **key**（字符串）；返回值不在映射表里会直接报错
- 变体：
    - 返回 `list[str]` → 同时走向**多个**目标（条件多分支）
    - 返回 `Command(goto=..., update=...)` → 边与 State 更新同时完成（`Command` 从 `langgraph.types` 导入，第 3 节详解）
- 调用时机：源节点执行完后、下一个 super-step 调度时调用；同一个路由函数每次执行都可能返回不同结果

条件分支是 LangGraph 最重要的控制流结构之一：本章第 2 节的循环（写诗 → 检查 → 重写）和第 2 章 3.1.2 的 plan → route → plan/answer 都是它实现的；ReAct 的「思考 → 行动 → 观察」循环同理。

下面给出与并行分支相同主题（主题创作）的条件分支案例。

In [ ]:
# 示例：条件分支 —— 同一组节点，运行时按 kind 选择走哪条边
# 与并行分支对比：并行是两条都跑，条件是根据 State 二选一

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    topic: str
    kind: str   # "joke" | "poem"，路由函数根据它选择下游


def joke_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」讲一个笑话")]
    )
    response.name = "joke"
    return {"messages": [response]}


def poem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」写一首诗")]
    )
    response.name = "poem"
    return {"messages": [response]}


# 路由函数：读 State，返回映射表的 key
def route(state: ChatState) -> str:
    return state["kind"]


builder = StateGraph(state_schema=ChatState)
builder.add_node("joke_node", joke_node)
builder.add_node("poem_node", poem_node)
# 条件边：直接从 START 出发，运行时根据 route 的返回值选择下游
# path_map 显式命名：路由函数返回值 → 目标节点 的映射表
builder.add_conditional_edges(START, route, path_map={"joke": "joke_node", "poem": "poem_node"})
builder.add_edge("joke_node", END)
builder.add_edge("poem_node", END)

graph = builder.compile()

# 同一组节点，两次 invoke 走不同路径
result = graph.invoke({"topic": "猫", "kind": "joke", "messages": [HumanMessage(content="请开始表演")]})
rprint([(m.name, m.content[:30]) for m in result["messages"] if m.name])

result = graph.invoke({"topic": "猫", "kind": "poem", "messages": [HumanMessage(content="请开始表演")]})
rprint([(m.name, m.content[:30]) for m in result["messages"] if m.name])

# 显示图结构：条件边从 START 出发，二选一
display(graph)

下面把条件分支扩展到**三个目标**：在主题创作中新增「国歌」节点，路由函数根据 `kind` 三选一。

In [ ]:
# 示例：条件分支三选一 —— 路由到国歌节点
# 与二选一相比只多了一个候选节点和一条映射项，路由函数不变

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    topic: str
    kind: str   # "joke" | "poem" | "anthem"


def joke_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」讲一个笑话")]
    )
    response.name = "joke"
    return {"messages": [response]}


def poem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」写一首诗")]
    )
    response.name = "poem"
    return {"messages": [response]}


# 新增的国歌节点：与 joke / poem 完全同构，只是提示词和 name 不同
def anthem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」创作一首国歌")]
    )
    response.name = "anthem"
    return {"messages": [response]}


def route(state: ChatState) -> str:
    return state["kind"]


builder = StateGraph(state_schema=ChatState)
builder.add_node("joke_node", joke_node)
builder.add_node("poem_node", poem_node)
builder.add_node("anthem_node", anthem_node)
# 三选一：路由函数返回值决定走 joke / poem / anthem 中的哪条
builder.add_conditional_edges(START, route, path_map={
    "joke": "joke_node",
    "poem": "poem_node",
    "anthem": "anthem_node",
})
builder.add_edge("joke_node", END)
builder.add_edge("poem_node", END)
builder.add_edge("anthem_node", END)

graph = builder.compile()

# 同一组节点，三种 kind 各走一条路径
for kind in ["joke", "poem", "anthem"]:
    result = graph.invoke({"topic": "猫", "kind": kind, "messages": [HumanMessage(content="请开始表演")]})
    rprint([(m.name, m.content[:30]) for m in result["messages"] if m.name])

# 显示图结构：条件边从 START 出发，三选一
display(graph)

路由函数不仅支持**三选一**（返回单个 key），还可以返回 `list[str]` **同时走向多个目标**——三选一 / 三选二 / 三选三，都由同一个映射表决定：

| 路由返回值 | 效果 | 对应写法 |
|---|---|---|
| `"joke"` | 3 选 1：只走一个节点 | 返回单个字符串 |
| `["joke", "poem"]` | 3 选 2：两个节点**同一 super-step 并行**执行 | 返回字符串列表 |
| `["joke", "poem", "anthem"]` | 3 选 3：三个节点全部执行（等价于并行分支的 `add_edge` 写法） | 返回完整列表 |

注意：多目标并行执行时，`messages` 由 `add_messages` 按**完成顺序**合并——谁先返回谁在前，顺序不保证（见下方输出）。

In [ ]:
# 示例：3 选 1 / 3 选 2 / 3 选 3 —— 路由函数返回 list[str] 同时走向多个目标
# 与上一个案例的区别：kinds 是列表，route_multi 原样返回它

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    topic: str
    kinds: list[str]   # 需要哪几个就放哪几个：["joke"] / ["joke","poem"] / 全部


def joke_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」讲一个笑话")]
    )
    response.name = "joke"
    return {"messages": [response]}


def poem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」写一首诗")]
    )
    response.name = "poem"
    return {"messages": [response]}


def anthem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」创作一首国歌")]
    )
    response.name = "anthem"
    return {"messages": [response]}


# 路由函数返回 list[str]：列表里的每个 key 都对应一个目标节点
def route_multi(state: ChatState) -> list[str]:
    return state["kinds"]


builder = StateGraph(state_schema=ChatState)
builder.add_node("joke_node", joke_node)
builder.add_node("poem_node", poem_node)
builder.add_node("anthem_node", anthem_node)
# 同一个映射表：单个字符串走一条，列表同时走多条
builder.add_conditional_edges(START, route_multi, path_map={
    "joke": "joke_node",
    "poem": "poem_node",
    "anthem": "anthem_node",
})
builder.add_edge("joke_node", END)
builder.add_edge("poem_node", END)
builder.add_edge("anthem_node", END)

graph = builder.compile()

# 3 选 1 / 3 选 2 / 3 选 3
for kinds in (["joke"], ["joke", "poem"], ["joke", "poem", "anthem"]):
    result = graph.invoke({"topic": "猫", "kinds": kinds, "messages": [HumanMessage(content="请开始表演")]})
    names = [m.name for m in result["messages"] if m.name]
    print(f"请求 {kinds} -> 实际执行: {names}")

# 显示图结构：条件边从 START 出发，三路候选，运行时可同时走多条
display(graph)

## 1.3 动态分支（Send API）

前两种分支的下游集合都在**编译期**确定。但有些场景编译期根本不知道下游有几个：比如「把任务拆成 N 个子任务并行处理」，N 是运行时才知道的。

`Send` 解决这个问题：Node 返回 `Command(goto=[Send(目标节点, 数据), ...])`，在**运行时动态创建并行任务**（map-reduce 的 map 部分）。每个 Send 相当于「向目标节点再投递一份输入」：

- `Send(node, payload)`：向 `node` 投递 `payload`，`payload` 会与当前 State 合并，成为该 node 的一次输入
- 返回几个 `Send`，目标节点就**并行**执行几次
- `Send` 从 `langgraph.types` 导入（与 `Command` 同源）

下面演示：planner 运行时把 3 个任务分发给 worker，worker 并行处理，结果用 reducer 汇总（纯 Python，不依赖 LLM）。

In [ ]:
# 示例：Send 动态分支 —— planner 运行时把 N 个任务分发给 worker 并行处理
# 与静态分支的区别：并行任务的数量（3 个 worker 实例）是运行时由 planner 决定的

from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, Send
from IPython.display import display


class SendState(TypedDict):
    tasks: list[str]  # 待处理任务
    results: Annotated[list[str], add]  # worker 结果，reducer 汇总


def planner(state: SendState) -> Command:
    # 每个任务向 worker 投递一份输入；worker 会并行执行 3 次
    return Command(goto=[Send("worker", {"task": t}) for t in state["tasks"]])


def worker(state) -> dict:
    # Send 的 payload 已合并进本次输入的 State，直接读 task
    return {"results": [f"处理完成: {state['task']}"]}


builder = StateGraph(state_schema=SendState)
builder.add_node("planner", planner)
builder.add_node("worker", worker)
builder.add_edge(START, "planner")
builder.add_edge("planner", END)  # planner 发完任务就结束自己的路径
builder.add_edge("worker", END)  # 每个 worker 实例结束后各自走向 END

graph = builder.compile()
display(graph)


print(graph.invoke({"tasks": ["a", "b", "c"], "results": []}))
# {'tasks': ['a', 'b', 'c'], 'results': ['处理完成: a', '处理完成: b', '处理完成: c']}

## 1.4 Defer 节点：并行分支的汇合点

`add_node(..., defer=True)`：让节点**推迟到调度器没有其它非 defer 工作可做时**再执行（`defer` 是 `StateNodeSpec` 上的参数，从 v0.6 起支持）。

- 适用场景：并行分支的**汇合点**（map-reduce、共识、多 agent 汇总）——聚合节点要等所有分支的结果都落地后再跑
- 语义：
  - defer 只是"推迟到无其它非 defer 工作"，**不是**"等所有祖先节点"；节点仍可能执行多次
  - 想让某个节点只在指定前提满足时跑，应显式用条件边 / `Command(goto=...)` 控制
- 对比（下面案例演示）：

| 写法 | 行为 |
|---|---|
| `add_node("aggregate", aggregate)` | 与并行分支**同一 super-step** 执行，读到的 State 里**没有**分支结果 |
| `add_node("aggregate", aggregate, defer=True)` | 分支先跑完（super-step 1），聚合节点在后续 super-step 执行，能看到**全部**结果 |

下面用主题创作演示：joke / poem 并行，聚合节点用 defer 等到两条回复都落地后再统计。

In [ ]:
# 示例：Defer 节点 —— 聚合节点推迟到并行分支全部完成后再执行
# 对比：不加 defer 时，aggregate 与 joke/poem 同一 super-step 并行，读不到分支结果

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    topic: str
    summary: str


def joke_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」讲一个笑话")]
    )
    response.name = "joke"
    return {"messages": [response]}


def poem_node(state: ChatState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content=f"根据主题「{state['topic']}」写一首诗")]
    )
    response.name = "poem"
    return {"messages": [response]}


# 聚合节点：defer=True → 推迟到没有其它非 defer 工作可做时才执行
def aggregate(state: ChatState) -> dict:
    names = [m.name for m in state["messages"] if m.name]
    print(f"[aggregate] 执行时看到的 AI 回复: {names}")
    return {"summary": f"共收到 {len(names)} 条创作: {names}"}


builder = StateGraph(state_schema=ChatState)
builder.add_node("joke_node", joke_node)
builder.add_node("poem_node", poem_node)
builder.add_node("aggregate", aggregate, defer=True)   # ← defer 节点
builder.add_edge(START, "joke_node")
builder.add_edge(START, "poem_node")
builder.add_edge(START, "aggregate")   # aggregate 也被 START 触发，但 defer 推迟执行
builder.add_edge("aggregate", END)

graph = builder.compile()

result = graph.invoke({"topic": "猫", "messages": [HumanMessage(content="请开始表演")]})
rprint(result["summary"])
# 输出示例: 共收到 2 条创作: ['joke', 'poem']   ← 两条并行回复都已落地

# 显示图结构：三条边从 START 出发，aggregate 是 defer 节点
display(graph)

# 2. 循环结构

循环是 LangGraph 最核心的能力：**边可以指回上游节点**，让 Graph 反复执行，直到条件边放行到 END。ReAct（思考 → 行动 → 观察 → 再思考）、生成-检查-重写都是循环。

两个关键点：

- 循环**必须**配合条件边：否则图会一直转下去。LangGraph 用 **recursion_limit** 兜底（默认 25 步），超过就抛 `GraphRecursionError`
- 每转一圈 State 都保留：`add_messages`、计数器 reducer 让 State 随循环累积——这是循环「越跑越接近目标」的前提

In [ ]:
# 示例：计数器循环 —— 转 3 圈自动退出（纯 Python，不依赖 LLM）

from operator import add
from typing import Annotated, TypedDict

from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph
from IPython.display import display

class LoopState(TypedDict):
    count: Annotated[int, add]


def inc(state: LoopState) -> dict:
    print(f"第 {state['count']} 圈")
    return {"count": 1}


def route(state: LoopState) -> str:
    return "loop" if state["count"] < 3 else END   # 满 3 圈放行到 END


builder = StateGraph(state_schema=LoopState)
builder.add_node("inc", inc)
builder.add_edge(START, "inc")
# 循环的关键：条件边的映射把 inc 指回自己；END 也放进映射表，route 返回 END 时直接结束
builder.add_conditional_edges("inc", route, {"loop": "inc", END: END})

graph = builder.compile()
display(graph)
print(graph.invoke({"count": 0}))
# 第 0 圈
# 第 1 圈
# 第 2 圈
# {'count': 3}


# recursion_limit 兜底：故意让循环永不退出，验证防死循环保护（默认上限 25 步）
def always_loop(state: LoopState) -> str:
    return "loop"


builder2 = StateGraph(state_schema=LoopState)
builder2.add_node("inc", inc)
builder2.add_edge(START, "inc")
builder2.add_conditional_edges("inc", always_loop, {"loop": "inc"})

try:
    builder2.compile().invoke({"count": 0}, config={"recursion_limit": 10})
except GraphRecursionError as e:
    print("触发了防死循环保护:", str(e)[:60])

In [ ]:
# 示例：LLM 自纠错循环 —— 写诗直到包含「月亮」，不合格就重写

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class PoemState(MessagesState):
    ok: bool


def write_poem(state: PoemState) -> dict:
    response = model.invoke(
        state["messages"] + [HumanMessage(content="写一首描述太阳或月亮的短诗")]
    )
    return {"messages": [response]}


def check(state: PoemState) -> dict:
    last = state["messages"][-1]
    ok = "月亮" in (last.content or "")
    print(f"检查: 诗中{'包含' if ok else '不含'}月亮")
    return {"ok": ok}


def route(state: PoemState) -> str:
    return "finish" if state["ok"] else "write_poem"   # 不合格 → 指回写诗节点重写


def finish(state: PoemState) -> dict:
    return {}


builder = StateGraph(state_schema=PoemState)
builder.add_node("write_poem", write_poem)
builder.add_node("check", check)
builder.add_node("finish", finish)
builder.add_edge(START, "write_poem")
builder.add_edge("write_poem", "check")
# 循环的关键：check 的条件边把 write_poem 指回自己，同时保留通往 finish 的分支
builder.add_conditional_edges("check", route, {"write_poem": "write_poem", "finish": "finish"})
builder.add_edge("finish", END)

result = builder.compile().invoke({"messages": [HumanMessage(content="开始")]})
rprint(result["messages"][-1].content)

In [ ]:
from operator import add
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    response: str
    stop: Literal["failure", "success"]


def llm_node(state: ChatState) -> dict:
    response = model.invoke(state["messages"])
    return {"response": response.content, "messages": [response]}


def check_node(state: ChatState) -> dict:
    words = ["股票", "摄影", "天气", "金融"]
    for word in words:
        if word in state["response"]:
            feedback = HumanMessage(
                content=f"上一段文字包含不允许内容「{word}」，请避开该主题重新写"
            )
            return {"stop": "failure", "messages": [feedback]}
    return {"stop": "success"}


def finish_node(state: ChatState) -> dict:
    return {}


def route_node(state: ChatState) -> str:
    return "llm_node" if state["stop"] == "failure" else "finish_node"


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("check_node", check_node)
builder.add_node("finish_node", finish_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "check_node")
builder.add_conditional_edges(
    "check_node", route_node, {"llm_node": "llm_node", "finish_node": "finish_node"}
)
builder.add_edge("finish_node", END)

graph = builder.compile()
display(graph)

result = graph.invoke(
    {
        "check_count": 0,
        "messages": [
            HumanMessage(
                content="请写一段简短文字，主题从金融、股票、摄影、天气、运动中选择一个"
            )
        ],
    }
)
rprint(result)

# 3. Command：跳转与更新二合一

前面 Node 都是「返回 dict 更新 State」，跳转交给边。`Command` 让 Node 在返回时**同时**写 State 和指定下一跳：

- `Command(goto=..., update=...)`：`goto` 指定下一个节点（覆盖图上的边），`update` 是 State 更新（正常走 reducer 合并）
- 适用场景：agent 手写循环（模型输出 Command 直接指定下一步并携带数据）、「边跑边决定」的复杂控制流
- `Command` 从 `langgraph.types` 导入

> 等价关系：`return Command(goto="b", update={...})` ≈ `return {...}` + 条件边路由到 b。Command 把两件事合成一个返回值，让 Node 自己决定走向。

In [ ]:
# 示例：Command —— check 节点同时更新 State 并指定下一跳（改写第 2 节的自纠错循环）
# 对比第 2 节案例：循环原来靠「stop 字段 + route 函数 + add_conditional_edges」实现，
# 这里 check_node 直接返回 Command(goto=..., update=...)：路由函数、stop 字段、条件边都不再需要

from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


class ChatState(MessagesState):
    response: str


def llm_node(state: ChatState) -> dict:
    response = model.invoke(state["messages"])
    return {"response": response.content, "messages": [response]}


def check_node(state: ChatState) -> Command:
    words = ["股票", "摄影", "天气", "金融"]
    for word in words:
        if word in state["response"]:
            feedback = HumanMessage(
                content=f"上一段文字包含不允许内容「{word}」，请避开该主题重新写"
            )
            # 不合格：Command 一步完成「更新 State（写回反馈）」+「指定下一跳（回 llm_node 重写）」
            return Command(goto="llm_node", update={"messages": [feedback]})
    # 合格：跳向 finish_node 收尾，无需再更新 State
    return Command(goto="finish_node")


def finish_node(state: ChatState) -> dict:
    return {}


builder = StateGraph(state_schema=ChatState)
builder.add_node("llm_node", llm_node)
builder.add_node("check_node", check_node)
builder.add_node("finish_node", finish_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "check_node")
# check_node 没有出边：走向完全由它返回的 Command.goto 决定（第 2 节的等价写法是条件边）
builder.add_edge("finish_node", END)

graph = builder.compile()
display(graph)

result = graph.invoke(
    {
        "messages": [
            HumanMessage(
                content="请写一段简短文字，主题从金融、股票、摄影、天气、运动中选择一个"
            )
        ],
    }
)
rprint(result)

# 4. 大模型配合工具调用

## 4.1 模拟工具调用

In [ ]:
from random import randint
from typing import Annotated, Literal
from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command
from rich import print as rprint

load_dotenv()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


@tool(parse_docstring=True)
def get_weather(city: str) -> str:
    """根据城市名称查询天气

    Args:
        city (str): 城市名称

    Returns:
        str: 城市天气情况
    """
    return f"{city} 的天气是晴天，温度 25°C"


@tool(parse_docstring=True)
def get_news(topic: Literal["科技", "体育", "娱乐"]) -> str:
    """根据主题查询新闻

    Args:
        topic (Literal["科技", "体育", "娱乐"]): 新闻主题

    Returns:
        str: 新闻内容
    """
    if topic == "科技":
        return "最新科技新闻：AI 技术正在快速发展。"
    elif topic == "体育":
        return "最新体育新闻：中国队在比赛中取得了胜利。"
    else:
        return "最新娱乐新闻：吴亦凡发布了新专辑。"


tools = [get_weather, get_news]
model_with_tools = model.bind_tools(tools)


class ChatState(MessagesState):
    user_input: str
    final_answer: str


def input_node(state: ChatState) -> dict:
    return {"messages": [HumanMessage(content=state["user_input"])]}


def llm_node(state: ChatState) -> Command[Literal["tool_node", "output_node"]]:
    ai_msg = model_with_tools.invoke(state["messages"])

    if ai_msg.tool_calls:
        next_node = "tool_node"
    else:
        next_node = "output_node"

    return Command(goto=next_node, update={"messages": [ai_msg]})


def tool_node(state: ChatState) -> dict:
    messages = state["messages"]
    last_msg = messages[-1]
    tool_calls = last_msg.tool_calls

    fail_prob = 6

    for call in tool_calls:
        if call["name"] == "get_weather":
            if randint(0, 9) < fail_prob:
                messages.append(
                    ToolMessage(
                        content="查询天气失败，请稍后再试", tool_call_id=call["id"]
                    )
                )
            else:
                result = get_weather.invoke(call["args"])
                messages.append(ToolMessage(content=result, tool_call_id=call["id"]))

        elif call["name"] == "get_news":
            if randint(0, 9) < fail_prob:
                messages.append(
                    ToolMessage(
                        content="查询新闻失败，请稍后再试", tool_call_id=call["id"]
                    )
                )
            else:
                result = get_news.invoke(call["args"])
                messages.append(ToolMessage(content=result, tool_call_id=call["id"]))

    return {"messages": messages}


def output_node(state: ChatState) -> dict:
    return {"final_answer": state["messages"][-1].content}


builder = StateGraph(state_schema=ChatState)
builder.add_node("input_node", input_node)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_node("output_node", output_node)

builder.add_edge(START, "input_node")
builder.add_edge("input_node", "llm_node")
builder.add_edge("tool_node", "llm_node")
builder.add_edge("output_node", END)

graph = builder.compile()

res = graph.invoke(
    {
        "user_input": "帮我查询一下上海的天气和娱乐新闻",
        "messages": [SystemMessage("如果工具调用失败，必须重新调用直到成功为止")],
    }
)
rprint(res)

## 4.2 reda、write、shell 工具调用

In [ ]:
from typing import Literal
from dotenv import load_dotenv
from IPython.display import display
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.types import Command
from rich import print as rprint
import subprocess
from pathlib import Path
from tavily import TavilyClient

load_dotenv()

# 自动读取 .env 中的 TAVILY_API_KEY
_tavily = TavilyClient()

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {
            "type": "disabled",
        },
    },
)


@tool(parse_docstring=True)
def read(file_path: str) -> str:
    r"""读取本地文本文件的完整内容，以 UTF-8 编码返回。

    适合读取代码、配置、文档等文本文件。不支持二进制文件（如图片、Excel）。
    文件不存在或读取失败时，返回以"读取失败:"开头的错误信息，而非抛出异常。

    Args:
        file_path: 文件路径，支持绝对路径（如 D:/Work/project/main.py）
            或相对于工作目录的相对路径（如 src/config.json）。分隔符用 / 或 \ 均可。

    Returns:
        文件全文内容；失败时返回"读取失败: <原因>"。
    """
    try:
        return Path(file_path).read_text(encoding="utf-8")
    except Exception as e:
        return f"读取失败: {e}"


@tool(parse_docstring=True)
def write(file_path: str, content: str) -> str:
    """将文本内容写入本地文件（UTF-8 编码），会覆盖文件原有内容。

    父目录不存在时会自动创建。适合保存代码、配置、笔记等文本文件。

    Args:
        file_path: 目标文件路径，支持绝对或相对路径，如 D:/Work/demo/output.txt。
        content: 要写入的完整文本内容，为空字符串时会清空文件。

    Returns:
        成功返回"写入成功: <文件路径>"；失败返回"写入失败: <原因>"。
    """
    try:
        Path(file_path).parent.mkdir(parents=True, exist_ok=True)
        Path(file_path).write_text(content, encoding="utf-8")
        return f"写入成功: {file_path}"
    except Exception as e:
        return f"写入失败: {e}"


@tool(parse_docstring=True)
def shell(command: str) -> str:
    """在 Windows 上执行 PowerShell 命令并返回输出结果，可运行任何命令行操作。

    例如：查看目录（Get-ChildItem）、运行 Python 脚本（python main.py）、
    安装依赖（pip install requests）、Git 操作（git status）等。
    命令超时时间为 30 秒，长时间运行的命令（如训练模型）请勿使用此工具。
    路径中的反斜杠会自动转换为正斜杠，无需额外处理。

    Args:
        command: 要执行的 PowerShell 命令，如 "Get-ChildItem D:/Work" 或
            "python -m pytest tests/ -v"。多个命令用 ; 分隔。

    Returns:
        成功时返回命令的标准输出（stdout），无输出时返回"(无输出)"；
        失败返回"错误: <stderr>"；超过 30 秒返回"命令执行超时"。
    """
    try:
        result = subprocess.run(
            ["powershell", "-Command", command],
            capture_output=True,
            text=True,
            timeout=30,
        )
        if result.returncode == 0:
            return result.stdout.strip() or "(无输出)"
        error = result.stderr.strip() or result.stdout.strip() or "未知错误"
        return f"错误: {error}"
    except subprocess.TimeoutExpired:
        return "命令执行超时"
    except Exception as e:
        return f"执行失败: {e}"


@tool(parse_docstring=True)
def web_search(query: str, max_results: int = 5) -> str:
    r"""联网搜索最新信息，返回与查询相关的网页标题、链接和内容摘要。

    适合查询实时或模型不知道的信息（新闻、最新版本、价格、文档等）。
    搜索失败时返回以"搜索失败:"开头的错误信息，而非抛出异常。

    Args:
        query: 搜索关键词或自然语言问题，如 "LangGraph 最新版本"。
        max_results: 返回结果数量，默认 5，建议 3~10。

    Returns:
        每条结果包含标题、链接和内容摘要的文本；失败时返回"搜索失败: <原因>"。
    """
    try:
        max_results = max(1, min(max_results, 10))
        res = _tavily.search(
            query=query, max_results=max_results, include_answer="basic"
        )
        parts = []
        if res.get("answer"):
            parts.append(f"参考答案: {res['answer']}")
        for r in res.get("results", []):
            parts.append(f"- {r['title']}\n  {r['url']}\n  {r['content']}")
        return "\n\n".join(parts) or "未找到相关结果"
    except Exception as e:
        return f"搜索失败: {e}"


@tool(parse_docstring=True)
def edit(file_path: str, old_text: str, new_text: str) -> str:
    r"""通过精确文本替换编辑文件，将 old_text 在文件中唯一匹配处替换为 new_text。

    适合对已有文件做局部修改（改一行、改函数名、改配置项等），不会动文件其他内容。
    old_text 必须与文件内容完全一致（包括缩进和换行）且在文件中唯一出现，
    不唯一时应扩大上下文范围使其唯一。一次只替换一处，修改多处请多次调用。
    文件不存在或编辑失败时，返回以"编辑失败:"开头的错误信息，而非抛出异常。

    Args:
        file_path: 目标文件路径，支持绝对或相对路径，如 D:/Work/project/main.py。
        old_text: 要替换的原文，必须与文件内容完全一致（含缩进和换行），
            如 "def add(a, b):\n    return a + b"。
        new_text: 替换后的新文本；传空字符串表示删除 old_text。

    Returns:
        成功返回"编辑成功: <文件路径>（替换 1 处）"；
        失败返回"编辑失败: <原因>"（含未找到、多处出现等具体提示）。
    """
    try:
        if not old_text:
            return "编辑失败: old_text 不能为空"

        content = Path(file_path).read_text(encoding="utf-8")

        count = content.count(old_text)
        if count == 0:
            return (
                "编辑失败: 未找到要替换的文本，old_text 必须与文件内容"
                "完全一致（包括缩进和换行），可先用 read 工具查看原文"
            )
        if count > 1:
            return (
                f"编辑失败: 要替换的文本出现 {count} 次，请扩大 old_text "
                "的上下文范围使其在文件中唯一"
            )
        if old_text == new_text:
            return "编辑失败: old_text 与 new_text 相同，文件不会有任何变化"

        Path(file_path).write_text(
            content.replace(old_text, new_text, 1), encoding="utf-8"
        )
        return f"编辑成功: {file_path}（替换 1 处）"
    except Exception as e:
        return f"编辑失败: {e}"


# 使用示例
tools = [read, write, shell, web_search, edit]
model_with_tools = model.bind_tools(tools)


class ChatState(MessagesState):
    user_input: str
    final_answer: str


def input_node(state: ChatState) -> dict:
    return {"messages": [HumanMessage(content=state["user_input"])]}


def llm_node(state: ChatState) -> Command[Literal["tool_node", "output_node"]]:
    ai_msg = model_with_tools.invoke(state["messages"])

    if ai_msg.tool_calls:
        next_node = "tool_node"
    else:
        next_node = "output_node"

    return Command(goto=next_node, update={"messages": [ai_msg]})


def tool_node(state: ChatState) -> dict:
    tool_map = {
        tool_.name: tool_
        for tool_ in (read, write, shell, web_search, edit)
    }
    tool_messages = []
    for call in state["messages"][-1].tool_calls:
        selected_tool = tool_map.get(call["name"])
        if selected_tool is None:
            result = f"工具调用失败: 未知工具 {call['name']}"
        else:
            result = selected_tool.invoke(call["args"])
        tool_messages.append(
            ToolMessage(content=result, tool_call_id=call["id"])
        )
    # 只返回新增消息，避免 MessagesState 重复合并完整历史。
    return {"messages": tool_messages}


def output_node(state: ChatState) -> dict:
    return {"final_answer": state["messages"][-1].content}


builder = StateGraph(state_schema=ChatState)
builder.add_node("input_node", input_node)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_node("output_node", output_node)

builder.add_edge(START, "input_node")
builder.add_edge("input_node", "llm_node")
builder.add_edge("tool_node", "llm_node")
builder.add_edge("output_node", END)

graph = builder.compile()
display(graph)
res = graph.invoke(
    {
        "user_input": "帮我搜一首 The Weekend 的歌词，写入桌面的 新建 文本文档.txt 文件中",
        "messages": [
            SystemMessage(content="如果工具调用失败，必须根据失败结果重新制定策略")
        ],
    }
)
rprint(res)
